In [ ]:
# ============================================================
# CELL 1: Install + imports
# v3 changes: lung-masked images, patient-level validation split,
# test set used exactly once at the end, faster in-memory pipeline
# ============================================================
!pip install -q kaggle opencv-python-headless torchxrayvision

import os, re, zipfile, json, time
import numpy as np
import tensorflow as tf
from pathlib import Path

# Let PyTorch (lung segmentation) and TensorFlow share the GPU
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f'TF: {tf.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ============================================================
# CELL 2: Kaggle auth + download
# Needs Colab secrets KAGGLE_USERNAME and KAGGLE_TOKEN (key icon, left sidebar)
# ============================================================
from google.colab import userdata

KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
KAGGLE_TOKEN    = userdata.get('KAGGLE_TOKEN')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_TOKEN}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
with zipfile.ZipFile('chest-xray-pneumonia.zip', 'r') as z:
    z.extractall('.')

DATA_DIR  = Path('chest_xray')
TRAIN_DIR = DATA_DIR / 'train'
VAL_DIR   = DATA_DIR / 'val'
TEST_DIR  = DATA_DIR / 'test'
print('✓ Dataset ready')

In [ ]:
# ============================================================
# CELL 3: Save location = this Colab machine + Hugging Face Hub (no Google Drive)
# Every important file is uploaded to the HF model repo as soon as it exists,
# so a disconnect or a Colab usage limit can't lose the trained model.
# Needs Colab secret HF_TOKEN with WRITE access.
# v3 file names never overwrite the deployed v2 files.
# ============================================================
from huggingface_hub import HfApi

SAVE_DIR = '/content/v3'
os.makedirs(SAVE_DIR, exist_ok=True)

HF_REPO = 'vijayesh684/chest-xray-densenet121'
hf_api  = HfApi(token=userdata.get('HF_TOKEN'))

# Fail now, not after an hour of training, if the token can't write
_who = hf_api.whoami()
_role = _who.get('auth', {}).get('accessToken', {}).get('role')
assert _role in ('write', 'fineGrained', None), f'HF_TOKEN must have write access (got role={_role!r})'
print(f"✓ HF login as {_who['name']} (token role: {_role}) -> {HF_REPO}")

def push(fname, folder=''):
    """Upload SAVE_DIR/fname to the HF model repo; retry a few times."""
    path_in_repo = f'{folder}/{fname}' if folder else fname
    for attempt in range(4):
        try:
            hf_api.upload_file(path_or_fileobj=f'{SAVE_DIR}/{fname}', path_in_repo=path_in_repo,
                               repo_id=HF_REPO, commit_message=f'v3: upload {path_in_repo}')
            print(f'  ↑ uploaded {path_in_repo} to {HF_REPO}')
            return
        except Exception as e:
            print(f'  upload failed ({e}); retrying...')
            time.sleep(15 * (attempt + 1))
    raise RuntimeError(f'could not upload {path_in_repo}')

print(f'✓ Save dir: {SAVE_DIR}')

In [ ]:
# ============================================================
# CELL 4: Config
# ============================================================
from tensorflow.keras import mixed_precision

IMG_SIZE   = 380
BATCH_SIZE = 16
VAL_SPLITS = 7   # hold out 1/7 (~14%) of PATIENTS for validation

# ~2x faster on a T4; the exported model is converted back to float32
mixed_precision.set_global_policy('mixed_float16')
print(f'IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, policy={mixed_precision.global_policy().name}')

In [ ]:
%%writefile lung_mask.py
"""
lung_mask.py — crop a chest X-ray down to the lungs.

Uses torchxrayvision's pretrained PSPNet (ChestX-Det) to segment the left and
right lungs, cleans each lung up (largest component + convex hull, so dense
pneumonia consolidation isn't cut out), dilates slightly, and blacks out
everything else: ECG leads, side markers, text, shoulders and image borders.

The same function must be used for training AND inference.
"""
import numpy as np
import cv2
import torch
import torchxrayvision as xrv

SEG_SIZE = 512
_model = None


def _seg_model():
    global _model
    if _model is None:
        _model = xrv.baseline_models.chestx_det.PSPNet()
        _model.eval()
        if torch.cuda.is_available():
            _model = _model.cuda()
    return _model


def _clean(m):
    """Largest connected component, then its convex hull."""
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)
    if n <= 1:
        return np.zeros_like(m, np.uint8)
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    comp = (lab == k).astype(np.uint8)
    cnts, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hull = np.zeros_like(comp)
    cv2.fillPoly(hull, [cv2.convexHull(np.vstack(cnts))], 1)
    return hull


def lung_masks(gray_batch):
    """
    gray_batch: list of HxW uint8 grayscale images (any size).
    Returns list of (mask HxW uint8 {0,1}, ok: bool).
    ok=False means segmentation found too little lung; mask is then all ones.
    """
    model = _seg_model()
    dev = next(model.parameters()).device
    x = np.stack([
        cv2.resize(g, (SEG_SIZE, SEG_SIZE), interpolation=cv2.INTER_AREA)
        for g in gray_batch
    ]).astype(np.float32)
    x = xrv.datasets.normalize(x, 255)[:, None]          # -> [-1024, 1024], Nx1xHxW
    with torch.no_grad():
        out = model(torch.from_numpy(x).to(dev))
    li, ri = model.targets.index("Left Lung"), model.targets.index("Right Lung")
    probs = torch.sigmoid(out[:, [li, ri]]).cpu().numpy()

    results = []
    pad = max(3, SEG_SIZE // 40)                          # ~2.5% margin
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * pad + 1, 2 * pad + 1))
    for g, p in zip(gray_batch, probs):
        m = _clean(p[0] > 0.5) | _clean(p[1] > 0.5)
        m = cv2.dilate(m, kernel)
        ok = m.mean() > 0.12                              # lungs are ~25-45% of the frame
        if not ok:
            m = np.ones_like(m)
        h, w = g.shape
        results.append((cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST), ok))
    return results


def mask_xray(gray, size):
    """Grayscale uint8 X-ray -> lung-only uint8 image of size x size, plus ok flag."""
    sq = cv2.resize(gray, (size, size), interpolation=cv2.INTER_AREA)
    (m, ok), = lung_masks([sq])
    return (sq * m).astype(np.uint8), ok

In [ ]:
# ============================================================
# CELL 5: Patient-level train/val split
# The official val folder has only 16 images, so pool train+val and hold
# out ~14% of PATIENTS (not images) for validation. The test set is NOT
# used for checkpointing, early stopping or the threshold any more.
# ============================================================
from sklearn.model_selection import StratifiedGroupKFold

def patient_id(path):
    name = path.name
    m = re.match(r'(person\d+)_', name)              # person1234_bacteria_567.jpeg
    if m:
        return m.group(1)
    m = re.match(r'((?:NORMAL2-)?IM-\d+)', name)     # IM-0115-0001.jpeg / NORMAL2-IM-0927-0001.jpeg
    return m.group(1) if m else name

def list_split(dirs):
    files, labels = [], []
    for d in dirs:
        for label, cls in enumerate(['NORMAL', 'PNEUMONIA']):
            for p in sorted((d / cls).glob('*.jp*g')):
                files.append(p)
                labels.append(label)
    return np.array(files), np.array(labels)

pool_files, pool_labels = list_split([TRAIN_DIR, VAL_DIR])
test_files, test_labels = list_split([TEST_DIR])
groups = np.array([patient_id(p) for p in pool_files])

sgkf = StratifiedGroupKFold(n_splits=VAL_SPLITS, shuffle=True, random_state=SEED)
tr_idx, va_idx = next(sgkf.split(pool_files, pool_labels, groups))
train_files, train_labels = pool_files[tr_idx], pool_labels[tr_idx]
val_files,   val_labels   = pool_files[va_idx], pool_labels[va_idx]

assert not set(groups[tr_idx]) & set(groups[va_idx]), 'patient leak between train and val'
for n, y in [('train', train_labels), ('val', val_labels), ('test', test_labels)]:
    print(f'{n:5s}: {len(y):5d} images | normal {int((y == 0).sum()):4d} | pneumonia {int((y == 1).sum()):4d}')
print(f'✓ {len(set(groups))} patients, none shared between train and val')

In [ ]:
# ============================================================
# CELL 6: Lung-mask every image once and cache it
# Blacks out everything outside the lungs (ECG leads, side markers,
# text, shoulders, borders) so the model can't learn those shortcuts.
# Also removes the per-epoch JPEG decode/resize that made training CPU-bound.
# ============================================================
import cv2, torch
import matplotlib.pyplot as plt
import lung_mask

CACHE = f'{SAVE_DIR}/masked_{IMG_SIZE}.npz'

def mask_files(files, bs=16):
    out, fails = np.zeros((len(files), IMG_SIZE, IMG_SIZE), np.uint8), 0
    for i in range(0, len(files), bs):
        sq = [cv2.resize(cv2.imread(str(f), cv2.IMREAD_GRAYSCALE), (IMG_SIZE, IMG_SIZE),
                         interpolation=cv2.INTER_AREA) for f in files[i:i + bs]]
        for j, (g, (m, ok)) in enumerate(zip(sq, lung_mask.lung_masks(sq))):
            out[i + j] = g * m
            fails += int(not ok)
        if i % 960 == 0:
            print(f'  {i}/{len(files)}')
    return out, fails

if os.path.exists(CACHE):
    d = np.load(CACHE)
    assert (d['y_train'] == train_labels).all() and (d['y_val'] == val_labels).all(), 'cache is from a different split'
    X_train, X_val, X_test = d['X_train'], d['X_val'], d['X_test']
    print('✓ Loaded cached masked images')
else:
    t0 = time.time()
    X_train, f1 = mask_files(train_files)
    X_val,   f2 = mask_files(val_files)
    X_test,  f3 = mask_files(test_files)
    print(f'✓ Masked {len(X_train) + len(X_val) + len(X_test)} images in {(time.time() - t0) / 60:.1f} min'
          f' | segmentation fallbacks (left unmasked): {f1 + f2 + f3}')
    np.savez(CACHE, X_train=X_train, X_val=X_val, X_test=X_test,
             y_train=train_labels, y_val=val_labels, y_test=test_labels)
    print(f'✓ Cached to {CACHE}')

# Free the segmentation model's GPU memory before training
lung_mask._model = None
torch.cuda.empty_cache()

idx = np.r_[np.where(train_labels == 0)[0][:4], np.where(train_labels == 1)[0][:4]]
fig, ax = plt.subplots(1, 8, figsize=(20, 3))
for a, i in zip(ax, idx):
    a.imshow(X_train[i], cmap='gray'); a.set_title(['normal', 'pneumonia'][train_labels[i]]); a.axis('off')
plt.show()

In [ ]:
# ============================================================
# CELL 7: Data pipeline + model
# Same head and layer names as v2, so the Space's Grad-CAM code still works.
# Augmentation runs on the GPU as part of the training model.
# No horizontal flip: the heart is on the left, flipping makes anatomy wrong.
# ============================================================
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.regularizers import l2
from sklearn.utils import class_weight

def to_ds(X, y, training):
    def gen():
        order = np.random.permutation(len(X)) if training else np.arange(len(X))
        for i in range(0, len(order), BATCH_SIZE):
            b = order[i:i + BATCH_SIZE]
            yield X[b], y[b].astype('float32')
    ds = tf.data.Dataset.from_generator(gen, output_signature=(
        tf.TensorSpec((None, IMG_SIZE, IMG_SIZE), tf.uint8),
        tf.TensorSpec((None,), tf.float32)))
    ds = ds.map(lambda x, y: (tf.image.grayscale_to_rgb(tf.cast(x[..., None], tf.float32)), y))
    return ds.prefetch(tf.data.AUTOTUNE)

def build_core(weights='imagenet'):
    base = EfficientNetB4(weights=weights, include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D(name='global_average_pooling2d')(x)
    x = layers.BatchNormalization(name='batch_normalization')(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=l2(1e-4), name='dense')(x)
    x = layers.Dropout(0.5, name='dropout')(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=l2(1e-4), name='dense_1')(x)
    x = layers.Dropout(0.3, name='dropout_1')(x)
    out = layers.Dense(1, activation='sigmoid', dtype='float32', name='dense_2')(x)
    return Model(inp, out, name='core'), base

augment = tf.keras.Sequential([
    layers.RandomRotation(10 / 360, fill_mode='constant'),
    layers.RandomTranslation(0.08, 0.08, fill_mode='constant'),
    layers.RandomZoom(0.12, fill_mode='constant'),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1, value_range=(0, 255)),
], name='augment')

core, base = build_core()
inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))
train_model = Model(inp, core(augment(inp)), name='train_model')

train_ds = to_ds(X_train, train_labels, training=True)
val_ds   = to_ds(X_val,   val_labels,   training=False)

cw = class_weight.compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weight_dict = dict(enumerate(cw))
print(f'Class weights: {class_weight_dict}')
print('✓ Model built')

In [ ]:
# ============================================================
# CELL 8: Phase 1 — Train head only (validation = held-out patients)
# ============================================================
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

base.trainable = False
train_model.compile(optimizer=Adam(1e-3), loss='binary_crossentropy',
                    metrics=['accuracy', AUC(name='auc')])

callbacks_p1 = [
    ModelCheckpoint(f'{SAVE_DIR}/v3_phase1_best.weights.h5', monitor='val_auc', mode='max',
                    save_best_only=True, save_weights_only=True, verbose=1),
    EarlyStopping(monitor='val_auc', mode='max', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
]

t0 = time.time()
history_p1 = train_model.fit(train_ds, validation_data=val_ds, epochs=10,
                             class_weight=class_weight_dict, callbacks=callbacks_p1)
print(f'Phase 1 took {(time.time() - t0) / 60:.1f} min')
push('v3_phase1_best.weights.h5', 'v3_checkpoints')

In [ ]:
# ============================================================
# CELL 9: Phase 2 — Fine-tune top 100 layers (BatchNorm stays frozen)
# ============================================================
base.trainable = True
for layer in base.layers[:-100]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

train_model.compile(optimizer=Adam(1e-5), loss='binary_crossentropy',
                    metrics=['accuracy', AUC(name='auc')])

callbacks_p2 = [
    ModelCheckpoint(f'{SAVE_DIR}/v3_phase2_best.weights.h5', monitor='val_auc', mode='max',
                    save_best_only=True, save_weights_only=True, verbose=1),
    EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-8, verbose=1),
]

t0 = time.time()
history_p2 = train_model.fit(train_ds, validation_data=val_ds, epochs=20,
                             class_weight=class_weight_dict, callbacks=callbacks_p2)
print(f'Phase 2 took {(time.time() - t0) / 60:.1f} min')
push('v3_phase2_best.weights.h5', 'v3_checkpoints')

In [ ]:
# ============================================================
# CELL 10: Threshold on VALIDATION, then evaluate on TEST exactly once
# ============================================================
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, f1_score

val_probs = core.predict(to_ds(X_val, val_labels, False), verbose=0).ravel()
fpr, tpr, thr = roc_curve(val_labels, val_probs)
THRESHOLD = float(thr[np.argmax(tpr - fpr)])
print(f'Threshold chosen on validation: {THRESHOLD:.4f} | val ROC-AUC {roc_auc_score(val_labels, val_probs):.4f}')

test_probs = core.predict(to_ds(X_test, test_labels, False), verbose=0).ravel()
pred = (test_probs > THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(test_labels, pred).ravel()
METRICS = {
    'accuracy':    round(100 * (tp + tn) / len(pred), 2),
    'roc_auc':     round(float(roc_auc_score(test_labels, test_probs)), 4),
    'sensitivity': round(100 * tp / (tp + fn), 2),
    'specificity': round(100 * tn / (tn + fp), 2),
    'f1_score':    round(float(f1_score(test_labels, pred)), 4),
}
print(f'\n{"=" * 50}\n  TEST RESULTS — EfficientNetB4 v3 (lung-masked)\n{"=" * 50}')
for k, v in METRICS.items():
    print(f'  {k:12s}: {v}')
print(f'  {"threshold":12s}: {THRESHOLD:.4f}\n  confusion   : TN={tn} FP={fp} FN={fn} TP={tp}\n{"=" * 50}')

with open(f'{SAVE_DIR}/metrics_v3.json', 'w') as f:
    json.dump(METRICS, f, indent=2)
with open(f'{SAVE_DIR}/optimal_threshold_v3.json', 'w') as f:
    json.dump({'threshold': THRESHOLD}, f)
push('metrics_v3.json'); push('optimal_threshold_v3.json')

In [ ]:
# ============================================================
# CELL 11: Export a float32 model (CPU-friendly for the Space)
# ============================================================
core.save_weights('/content/core.weights.h5')
mixed_precision.set_global_policy('float32')
# Must be built with weights='imagenet': Keras adds an extra input-rescaling
# layer ONLY in that case. weights=None silently changes the model's maths.
export_model, _ = build_core(weights='imagenet')
export_model.load_weights('/content/core.weights.h5')

check = X_val[:32]
x = tf.image.grayscale_to_rgb(tf.cast(check[..., None], tf.float32))
diff = np.abs(export_model.predict(x, verbose=0) - core.predict(x, verbose=0)).max()
print(f'float32 vs mixed-precision max prediction difference: {diff:.4f}')
assert diff < 0.02

export_model.save(f'{SAVE_DIR}/best_model_v3.keras')
with open(f'{SAVE_DIR}/optimal_threshold_v3.json', 'w') as f:
    json.dump({'threshold': THRESHOLD}, f)
with open(f'{SAVE_DIR}/metrics_v3.json', 'w') as f:
    json.dump(METRICS, f, indent=2)
print(f'✓ Saved best_model_v3.keras, optimal_threshold_v3.json, metrics_v3.json to {SAVE_DIR}')
push('best_model_v3.keras')

In [ ]:
# ============================================================
# CELL 12: Grad-CAM check — does attention now stay in the lungs?
# Same method as the Space: predicted class, pre-sigmoid logit.
# ============================================================
m = tf.keras.models.load_model(f'{SAVE_DIR}/best_model_v3.keras')
eb4 = m.get_layer('efficientnetb4')
grad_model = tf.keras.Model(eb4.input, [eb4.get_layer('top_activation').output, eb4.output])
d2 = m.get_layer('dense_2')

def gradcam(img_u8):
    x = tf.image.grayscale_to_rgb(tf.cast(img_u8[None, ..., None], tf.float32))
    with tf.GradientTape() as tape:
        conv, feat = grad_model(x, training=False)
        tape.watch(conv)
        h = m.get_layer('global_average_pooling2d')(feat)
        h = m.get_layer('batch_normalization')(h, training=False)
        h = m.get_layer('dense_1')(m.get_layer('dense')(h))
        logit = (tf.matmul(h, d2.kernel) + d2.bias)[:, 0]
        prob = float(tf.sigmoid(logit)[0])
        target = logit if prob > THRESHOLD else -logit
    g = tape.gradient(target, conv)
    cam = tf.nn.relu(tf.squeeze(conv[0] @ tf.reduce_mean(g, axis=(0, 1, 2))[..., None])).numpy()
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC).clip(0)
    return cam / cam.max() if cam.max() > 0 else cam, prob

# Share of attention that falls inside the lungs, over 120 test images
rng = np.random.default_rng(SEED)
inside = []
for i in rng.choice(len(X_test), 120, replace=False):
    cam, _ = gradcam(X_test[i])
    inside.append(cam[X_test[i] > 0].sum() / max(cam.sum(), 1e-9))
print(f'Attention inside the lungs: mean {np.mean(inside) * 100:.1f}% | median {np.median(inside) * 100:.1f}%')
with open(f'{SAVE_DIR}/gradcam_check_v3.json', 'w') as f:
    json.dump({'inside_lungs_mean': float(np.mean(inside)), 'inside_lungs_median': float(np.median(inside)),
               'n_images': len(inside)}, f, indent=2)
push('gradcam_check_v3.json')

idx = np.r_[np.where(test_labels == 0)[0][:4], np.where(test_labels == 1)[0][:4]]
fig, ax = plt.subplots(1, 8, figsize=(20, 3))
for a, i in zip(ax, idx):
    cam, p = gradcam(X_test[i])
    a.imshow(X_test[i], cmap='gray'); a.imshow(cam, cmap='jet', alpha=0.4)
    a.set_title(f"{['normal', 'pneumonia'][test_labels[i]]} | {p:.2f}"); a.axis('off')
plt.show()

In [ ]:
# ============================================================
# CELL 13: Confirm everything reached the HF Hub
# ============================================================
files = set(hf_api.list_repo_files(HF_REPO))
need = ['best_model_v3.keras', 'optimal_threshold_v3.json', 'metrics_v3.json', 'gradcam_check_v3.json']
for n in need:
    print(f"{'✓' if n in files else '✗ MISSING'} {n}")
assert all(n in files for n in need)
print('All v3 files are safe on the Hub. The Space still serves v2 until its code is updated.')